# 22 · Réseaux récurrents (RNN, LSTM, GRU) — solutions (notebook exécuté)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cemah2/workbookIA/blob/main/chapitres/ch22_rnn/05_solutions.ipynb)

Les réponses des exercices, exécutées. Les démarches détaillées (le *pourquoi*, les erreurs fréquentes, les variantes) sont dans `05_solutions.md`. Les cellules marquées `answer` enregistrent les réponses vérifiées par `wb.check` (`tools/build_answers.py`).

| Partie | ID | Titre | Type | ★ | ⏱️ |
|---|---|---|---|---|---|
| 0 | 22.Q1 à 22.Q12, 22.R1 à 22.R3, 22.1 à 22.5, 22.8 | vérifier tes réponses courtes | 🧠 🔁 ✏️ 🧮 | ★ à ★★ | |
| A | 22.12 à 22.14 | ta librairie sequences.py : vocabulaire, encodage et fenêtres ; un MLP sur des fenêtres mélangées (*prochaine session de génération*) | | | |
| B | 22.15 à 22.19 | ta librairie rnn.py : les cellules RNN, LSTM et GRU, comparées à PyTorch ; la disparition du gradient mesurée avec autograd (*prochaine session de génération*) | | | |
| C | 22.20 à 22.24 | l'échantillonnage avec température, une série temporelle piégée, les formes des RNN profonds et bidirectionnels, les taches solaires face à la persistance (*prochaine session de génération*) | | | |
| D | 22.25 à 22.29 | le générateur de Holmes, sa loss et sa perplexité, Holmes contre Verne, le CNN-LSTM, le défi des taches solaires (*prochaine session de génération*) | | | |

In [1]:
# ⚙️ SETUP: run this cell first (on Colab or on your computer)
FAST_MODE = True  # True = quick version (CPU, a few minutes); False = full version
SEED = 42

import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/cemah2/workbookIA.git"
DRIVE_DIR = Path("/content/drive/MyDrive/workbookIA")

try:
    import google.colab  # noqa: F401  (this module only exists on Colab)
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")
    subprocess.run(["git", "config", "--global", "--add", "safe.directory", str(DRIVE_DIR)])
    if not (DRIVE_DIR / ".git").exists():
        subprocess.run(["git", "clone", REPO_URL, str(DRIVE_DIR)], check=True)
    else:
        # Get the new chapters. Your own work only lives in mon_travail/, which Claude never
        # touches, so "rebase + autostash" keeps it safe (even with local commits).
        identity = subprocess.run(["git", "-C", str(DRIVE_DIR), "config", "user.email"],
                                  capture_output=True).returncode == 0
        who = [] if identity else ["-c", "user.name=Workbook", "-c", "user.email=workbook@localhost"]
        pull = subprocess.run(["git", *who, "-C", str(DRIVE_DIR), "pull", "--rebase", "--autostash"],
                              capture_output=True, text=True)
        conflicts = subprocess.run(["git", "-C", str(DRIVE_DIR), "diff", "--name-only", "--diff-filter=U"],
                                   capture_output=True, text=True).stdout.split()
        if conflicts:  # a file changed outside mon_travail/ AND by Claude: git leaves conflict markers in it
            print("⚠️ Tes modifications de " + ", ".join(conflicts) + " (hors de mon_travail/) entrent en conflit "
                  "avec la nouvelle version ; elles sont gardées dans `git stash`. Pour reprendre la version du "
                  "dépôt : git restore --source=HEAD --staged --worktree <fichier> (00_setup/COLAB.md, « Problèmes "
                  "fréquents »).")
        elif pull.returncode != 0:
            print("⚠️ git pull a échoué (voir 00_setup/COLAB.md, « Problèmes fréquents ») :\n" + pull.stderr)
    ROOT = DRIVE_DIR
else:
    ROOT = Path(os.environ.get("WB_ROOT", Path.cwd())).resolve()
    while not (ROOT / "src" / "wb").exists() and ROOT != ROOT.parent:
        ROOT = ROOT.parent
    if not (ROOT / "src" / "wb").exists():
        raise FileNotFoundError("Dépôt introuvable : ouvre ce notebook depuis le dossier du workbook.")

os.environ["WB_ROOT"] = str(ROOT)
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

import wb  # noqa: E402

cfg = wb.setup(seed=SEED, fast=FAST_MODE)
FAST_MODE = cfg.fast
mylearn = wb.load_mylearn("ref")  # reference implementation

🧪 Deep Learning Workbook : environnement
   Plateforme : Local (Linux x86_64, 2 CPU)
   Python 3.13.13 | numpy 2.1.3 | pandas 2.2.3 | matplotlib 3.10.0 | scikit-learn 1.6.1 | scipy 1.16.3 | torch 2.11.0+cpu | torchvision 0.26.0+cpu
   Device : cpu | Graine : 42 | FAST_MODE : True


## Objectifs et rappel express

- Découper une série ou un texte en fenêtres, compter les exemples, et valider une prévision sans fuite de données.
- Programmer en NumPy, dans `mylearn.rnn`, les cellules RNN, LSTM et GRU, et les comparer à `nn.RNN`, `nn.LSTM` et `nn.GRU`.
- Mesurer la disparition du gradient, puis entraîner un LSTM qui prévoit les taches solaires et un autre qui écrit du Holmes caractère par caractère.

**Rappel express.** Cellule RNN : $h_t = \tanh(x_t W_{ih} + b_{ih} + h_{t-1} W_{hh} + b_{hh})$. LSTM (portes $i, f, g, o$) : $c_t = f_t \odot c_{t-1} + i_t \odot g_t$, $h_t = o_t \odot \tanh(c_t)$. GRU (portes $r, z, n$) : $h_t = (1 - z_t) \odot n_t + z_t \odot h_{t-1}$. Paramètres d'une couche PyTorch : $k\,(DH + H^2 + 2H)$, $k = 1, 4, 3$. Fenêtres : $N = \lfloor (T - w - h)/s \rfloor + 1$. Température : $p = \mathrm{softmax}(z/T)$.

## Partie 0 · Vérifier tes réponses courtes (quiz, rappels, ✏️ 22.1 à 22.5 et 🧮 22.8)

Fais d'abord les quiz, les rappels et les exercices de `02_exercices.md` **sur papier**, dans ta copie de `06_mes_reponses.md`. Reporte ensuite chaque réponse courte ici : **la valeur** que tu as trouvée, pas l'expression Python, sinon tu ne vérifies rien. Un nombre : `42` ou `0.3` (en Python, le séparateur décimal est un **point** ; `0,3` sans guillemets serait un couple de deux nombres) ; un vrai ou faux : `True` ou `False` ; un choix : la lettre seule, entre guillemets (`"E"`) ; une forme : entre guillemets (`"(3, 5, 1)"`) ; un mot : entre guillemets (`"fit"`). Arrondis comme l'énoncé le demande, et seulement à la fin du calcul. Les réponses pas encore remplies affichent ⏳. Les questions « dans ta copie », les preuves ∂ 22.6 et 22.7, la réflexion (🗣️ 22.9, ⚖️ 22.10, 📄 22.11) et l'entretien se corrigent avec `05_solutions.md`.

In [2]:
# The short answers only use the numbers of their statements (02_exercices.md)
import math

import numpy as np


def sigmoid(z):
    return 1 / (1 + math.exp(-z))


def n_windows(T, w, h=1, s=1):
    """Number of windows of w values whose target lies h steps after the window (starts s apart)."""
    return (T - w - h) // s + 1


def n_recurrent(d, h, blocks):
    """Parameters of one PyTorch recurrent layer: blocks of H columns, two bias vectors."""
    return blocks * (d * h + h * h + 2 * h)


def softmax_t(z, t):
    e = np.exp((np.asarray(z, float) - max(z)) / t)
    return e / e.sum()


# Ex 22.2: a scalar RNN cell h_t = tanh(u x_t + w h_{t-1} + b), y_t = v h_t
def scalar_rnn(xs, u=1.0, w=0.5, b=0.0, h=0.0):
    states = []
    for x in xs:
        h = math.tanh(u * x + w * h + b)
        states.append(h)
    return states


H_222 = scalar_rnn([2, 0, -1])
H_REV_222 = scalar_rnn([-1, 0, 2])

# Ex 22.4: one LSTM step, gates in the order i, f, g, o
W_IH_224, W_HH_224, B_224 = [-0.5, 1, 1, -1], [1, -2, 0, 3], [0, 1, 0, 0]
A_224 = [W_IH_224[k] * 1 + W_HH_224[k] * 0.5 + B_224[k] for k in range(4)]
I_224, F_224, G_224, O_224 = sigmoid(A_224[0]), sigmoid(A_224[1]), math.tanh(A_224[2]), sigmoid(A_224[3])
C_224 = F_224 * 2 + I_224 * G_224
H_224 = O_224 * math.tanh(C_224)
C_OPEN_224 = sigmoid(A_224[1] + 20) * 2 + sigmoid(A_224[0] - 20) * G_224

# Ex 22.5: the logits of « k », « e », « a » after « Sherloc »
Z_225 = [3.0, 1.0, 0.5]

# Ex 22.8: the CNN-LSTM of the book (128 x 128 frames, 8 filters 5 x 5, stride 2, padding 2, LSTM 128, 9 classes)
SIDE_228 = (128 + 2 * 2 - 5) // 2 + 1
D_228 = SIDE_228 * SIDE_228 * 8
CONV_228 = 8 * (5 * 5 + 1)
LSTM_228 = n_recurrent(D_228, 128, 4)
DENSE_228 = 128 * 9 + 9
MACS_228 = D_228 * 4 * 128

**22.Q1 — Pourquoi un réseau dense ignore l'ordre**

In [3]:
wb.record("22.Q1a", True, mistakes={"une permutation FIXE des entrées : le réseau peut-il permuter ses poids de la même façon ?": False})
wb.record("22.Q1b", "B", mistakes={"une couche dense ne trie rien : elle multiplie chaque entrée par son poids": "A",
          "la loss dépend bien des entrées, à travers les prédictions": "C",
          "c'est le RNN qui garde un état, pas la couche dense": "D"})
wb.record("22.Q1c", False, mistakes={"si l'ordre change d'une fenêtre à l'autre, une entrée porte-t-elle encore toujours la même position ?": True})
wb.record("22.Q1d", "B", mistakes={"des nombres sans lien entre eux ne forment pas une séquence": "A",
          "la quantité de données ne fait pas une séquence": "C", "les labels ne font pas une séquence": "D"})
wb.record("22.Q1e", True, mistakes={"écris les positions des fenêtres qui démarrent en 0 et en 1": False})

WB_ANSWER {"hash": "d8293619dc0f06fde351660650bfffbbf0b507cd82467f521ec2316beffdb41e", "id": "22.Q1a", "kind": "bool", "mistakes": {"577a7c4dc2ba2a784a5af12e3d21dfaad7742273de71a59e8df0966d700b4af7": "une permutation FIXE des entrées : le réseau peut-il permuter ses poids de la même façon ?"}}
📝 Ex 22.Q1a : réponse enregistrée (bool).
WB_ANSWER {"hash": "3cc7dc91d2fa894f31008ff4aa2b7dd192c976506e5e5e2409450fcf1f6b5370", "id": "22.Q1b", "kind": "str", "mistakes": {"2837ec752f979bc3e0e2105b9d93ddc5008245b627df29f791ca7d591f8591a0": "c'est le RNN qui garde un état, pas la couche dense", "305b588926a38fd29ce590132253488ea0f22f7866b082498551235799a94223": "la loss dépend bien des entrées, à travers les prédictions", "a85312deb378aa44b11ea012c7dc553d019f645237070871a779d6bc4d449b14": "une couche dense ne trie rien : elle multiplie chaque entrée par son poids"}}
📝 Ex 22.Q1b : réponse enregistrée (str).
WB_ANSWER {"hash": "dddb0cb7c0c228676d0f16fd854f4402b1b358e186c140804fbca2160a0ff49a", "id"

**22.Q2 — État ou poids : qu'est-ce qui change en production ?**

In [4]:
wb.record("22.Q2a", False, mistakes={"sans optimiseur, qu'est-ce qui modifierait les poids ?": True})
wb.record("22.Q2b", True, mistakes={"relis le §22.3 : que calcule la cellule à chaque nouvelle entrée ?": False})
wb.record("22.Q2c", True, mistakes={"mêmes poids, mêmes entrées, même état de départ : qu'est-ce qui pourrait différer ?": False})
wb.record("22.Q2d", False, mistakes={"après la phrase A, l'état de départ de B est-il encore nul ?": True})
wb.record("22.Q2e", "B", mistakes={"les poids ne changent pas d'une entrée à l'autre en production": "A",
          "les gradients ne servent qu'à l'entraînement": "C", "le learning rate est un hyperparamètre fixe": "D"})

WB_ANSWER {"hash": "545d59f07f237da45349a0bbf96696253ed0007cb3b46b5333d97aaf856a4509", "id": "22.Q2a", "kind": "bool", "mistakes": {"f44cff714dafa5a636c6b2004761d0138514bc8793b53d1c883a9e52f3f3908f": "sans optimiseur, qu'est-ce qui modifierait les poids ?"}}
📝 Ex 22.Q2a : réponse enregistrée (bool).
WB_ANSWER {"hash": "f71dc6ae0bd888635a9a663757183ecad26524db19ca3d273f202b7916a3bb33", "id": "22.Q2b", "kind": "bool", "mistakes": {"5dd6ed674d9ec36ff01ec3f726ddcb424bbb273efda37434b85e76a42e813174": "relis le §22.3 : que calcule la cellule à chaque nouvelle entrée ?"}}
📝 Ex 22.Q2b : réponse enregistrée (bool).
WB_ANSWER {"hash": "76c2c5c43ef80c308b2127082fda9d2f6b668402d9c1c8393cb96ba1a8f63c43", "id": "22.Q2c", "kind": "bool", "mistakes": {"f538c4ada8efec8e2211d1a2a50635685403f71c10b9a482368cf9366865a752": "mêmes poids, mêmes entrées, même état de départ : qu'est-ce qui pourrait différer ?"}}
📝 Ex 22.Q2c : réponse enregistrée (bool).
WB_ANSWER {"hash": "a785df1316fe9c0a670ece6f6d3a1827ecb7

**22.Q3 — Délai, déroulement et pas de temps**

In [5]:
wb.record("22.Q3a", "B", mistakes={"l'activation est dans la cellule, pas dans le petit carré": "A",
          "le biais est un poids de la cellule": "C", "le dropout n'apparaît pas dans ce schéma": "D"})
wb.record("22.Q3b", 1, mistakes={"les copies du schéma déroulé ont-elles leurs propres poids ?": 12})
wb.record("22.Q3c", 12, mistakes={"une application de la cellule par pas de temps": 1})
wb.record("22.Q3d", False, mistakes={"relis le paragraphe « Enroulé et déroulé » : les copies partagent-elles leurs poids ?": True})
wb.record("22.Q3e", 2, mistakes={"une matrice pour l'entrée, une pour l'état précédent": 1,
          "4, c'est le nombre de blocs d'un LSTM ; une cellule RNN n'en a qu'un": 4})

WB_ANSWER {"hash": "a77a241d980dc44f554562eb199a31938fd0c099c439b63c78a3e8376c5b8d74", "id": "22.Q3a", "kind": "str", "mistakes": {"2bfac4e714110dfcc57724cb5ab826bb2d783dd928f3bb7b93a42f5424f2ef9e": "le dropout n'apparaît pas dans ce schéma", "3765b56658b33f7214401607a4b543c65c1fd1010c85424f6f5ee9bacf9dad63": "l'activation est dans la cellule, pas dans le petit carré", "c72a9593042a53ab9b435c4bd9fe903e15122f68e0e6259c9253ec5a8279a538": "le biais est un poids de la cellule"}}
📝 Ex 22.Q3a : réponse enregistrée (str).
WB_ANSWER {"hash": "9f4bc4e56487bc76085dc6a1f2f991a4eee0bc0d72c0da1c21e1777dd6c219b9", "id": "22.Q3b", "kind": "int", "magnitude": 0, "mistakes": {"0b4496389b65a061a9acf4819e81ff33630ed85088fd7461d8d9c921d6569274": "les copies du schéma déroulé ont-elles leurs propres poids ?"}, "sign": 1}
📝 Ex 22.Q3b : réponse enregistrée (int).
WB_ANSWER {"hash": "a3182a481d9b00ad24abdd748fcc85594eb45b62264271305766a316b7133874", "id": "22.Q3c", "kind": "int", "magnitude": 1, "mistakes": {

**22.Q4 — Entrée one-hot, taille de l'état et sens de ses valeurs**

In [6]:
wb.record("22.Q4a", 3000, mistakes={"la longueur du vecteur, pas son nombre de 1": 1})
wb.record("22.Q4b", 1, mistakes={"un vecteur one-hot n'a qu'un seul 1": 3000})
wb.record("22.Q4c", False, mistakes={"W_ih a la forme (D, H) : D et H sont-ils liés ?": True})
wb.record("22.Q4d", 30 * 64, mistakes={"un poids par couple (entrée, valeur d'état) : un produit": 30 + 64,
          "c'est la taille de W_hh, (H, H)": 64 * 64, "sans les biais": 30 * 64 + 64})
wb.record("22.Q4e", "B", mistakes={"personne ne fixe le contenu de l'état à l'avance": "A",
          "l'état résume ce qui a été lu, il ne range pas les caractères un par un": "C",
          "la loss n'est pas stockée dans l'état": "D"})

WB_ANSWER {"hash": "9db2183c976b6e5661f3cae0f1d2af1da28e00fe1d43cafbad0331403be388ef", "id": "22.Q4a", "kind": "int", "magnitude": 3, "mistakes": {"91f7b643698e1d08e816c5868d00ee3cd15377a0afe10229dacf83fc8807a151": "la longueur du vecteur, pas son nombre de 1"}, "sign": 1}
📝 Ex 22.Q4a : réponse enregistrée (int).
WB_ANSWER {"hash": "846675236a12439199c60d2d1a184bc4d711ad1dddc05b1475d32d2345e6ae58", "id": "22.Q4b", "kind": "int", "magnitude": 0, "mistakes": {"ade1abe9f555242798707f1d4c1fe30a8db8d5f46fdff97ea46461a1898af0ce": "un vecteur one-hot n'a qu'un seul 1"}, "sign": 1}
📝 Ex 22.Q4b : réponse enregistrée (int).
WB_ANSWER {"hash": "8f9d782a7d5c7f1b50b717634cef0609cc0ca341d457581a30ba35f4df3ee021", "id": "22.Q4c", "kind": "bool", "mistakes": {"bf233d2a84ad94622901fc9262c027652edd9fcb34d94320d06f47811f096fd5": "W_ih a la forme (D, H) : D et H sont-ils liés ?"}}
📝 Ex 22.Q4c : réponse enregistrée (bool).
WB_ANSWER {"hash": "7a970a9cfd63b17b4e2b90fee858e0a9459cd90b0f895cfb7bd0543310ec68d0

**22.Q5 — (échantillons, pas de temps, features) : la bonne organisation**

In [7]:
wb.record("22.Q5a", "(10, 6, 2)", mistakes={"c'est l'ordre par défaut de PyTorch ; ici, batch_first=True": "(6, 10, 2)",
          "(N, T, D) : les pas de temps avant les features": "(10, 2, 6)",
          "les nuits ne sont pas mesurées : une séquence par journée": "(1, 60, 2)"})
wb.record("22.Q5b", "(6, 10, 2)", mistakes={"c'est la forme avec batch_first=True": "(10, 6, 2)",
          "(T, N, D) : les features en dernier": "(6, 2, 10)"})
wb.record("22.Q5c", False, mistakes={"reshape relit les nombres dans le même ordre : essaie sur np.arange(12).reshape(2, 3, 2)": True})
wb.record("22.Q5d", "B", mistakes={"les nuits manquent : 18 h ne précède pas directement 8 h": "A",
          "une mesure seule n'est pas une séquence": "C", "les mesures successives sont des pas de temps, pas des features": "D"})
wb.record("22.Q5e", 1, mistakes={"le deuxième nombre de (N, T, D)": 6})

WB_ANSWER {"hash": "854f65e0116cc1715ef0cf4a0a332f1da9048fb63dd38605568d8d746d515a8a", "id": "22.Q5a", "kind": "str", "mistakes": {"1e8642dfdd20d6cd0e02f7a49282c8ba51e923a7a029e406a1113650da28cb62": "c'est l'ordre par défaut de PyTorch ; ici, batch_first=True", "21bcf0f188a69682ecac854aefa58351bc9bbc20568393b595102a6381f013e6": "les nuits ne sont pas mesurées : une séquence par journée", "66f89b6ac5362e790738f825e6e7662fd28279cdfe9f9e45d5a1ef8ac3743953": "(N, T, D) : les pas de temps avant les features"}}
📝 Ex 22.Q5a : réponse enregistrée (str).
WB_ANSWER {"hash": "816c29efaada8d8b68146a9c23ef97e909c1c1422d56ca63ceadb23e504f2c8b", "id": "22.Q5b", "kind": "str", "mistakes": {"232802b8cbe442b1031eba9541a659c69d5d0bf44c91952cdc2151abaad97973": "c'est la forme avec batch_first=True", "a16fb268cda50b3a1f15c9d842788501b1aa27146fedae63d6bd173bf252d5cd": "(T, N, D) : les features en dernier"}}
📝 Ex 22.Q5b : réponse enregistrée (str).
WB_ANSWER {"hash": "bf34b2ba5d002b60987cfdf77c5d16f2af9d17cc

**22.Q6 — BPTT, gradients évanescents et explosifs, dépendances longues**

In [8]:
wb.record("22.Q6a", "B", mistakes={"la BPTT calcule le gradient ; l'optimiseur s'en sert ensuite": "A",
          "l'initialisation précède l'entraînement": "C", "le dropout est une régularisation": "D"})
wb.record("22.Q6b", "C", mistakes={"0,9⁵⁰ = e^{50 ln 0,9}, avec ln 0,9 ≈ −0,105": "A", "calcule e^{−5,3}": "B",
          "e^{−5,3} vaut un peu plus de 0,005": "D"})
wb.record("22.Q6c", "C", mistakes={"1,1⁵⁰ = e^{50 ln 1,1}, avec ln 1,1 ≈ 0,095": "A", "calcule e^{4,8}": "B",
          "e^{4,8} vaut un peu plus de 100": "D"})
wb.record("22.Q6d", False, mistakes={"le clipping agrandit-il un gradient trop petit ?": True})
wb.record("22.Q6e", "B", mistakes={"il s'agit de la séquence, pas des poids": "A", "il s'agit d'une entrée lointaine": "C",
          "relis l'encadré « Le livre, corrigé » du §22.6 : mémoire ou apprentissage ?": "D"})

WB_ANSWER {"hash": "ad82e6809b9cbb58360f998e176cc411b8f3d32ecb130cb94210be3ca850f5cf", "id": "22.Q6a", "kind": "str", "mistakes": {"68ae3d47ccc3678d24dd47496ac4b027fb1254f9936542366b785c455a012c87": "l'initialisation précède l'entraînement", "69ca6fe93c9d29b24b0af88791d1a6fd42c2329637cd4504885e1840f826cbee": "la BPTT calcule le gradient ; l'optimiseur s'en sert ensuite", "d745f8dd368972c969ecdbc2c13adb8dbdbbd852d8cf203aab9a56781afcdeeb": "le dropout est une régularisation"}}
📝 Ex 22.Q6a : réponse enregistrée (str).
WB_ANSWER {"hash": "e0006a7a28f5b82c1a6a5df13b384f4513dfdff3e763872e2a0e995fa96bbd53", "id": "22.Q6b", "kind": "str", "mistakes": {"6a63fd460e2780274ac8db9ef569467027177d176d8c63a53129bb5475a58f15": "calcule e^{−5,3}", "92475bbb8dd8a6fb3b4abff1a89d19f890357d27ada0237c6f38760d0d9001f5": "0,9⁵⁰ = e^{50 ln 0,9}, avec ln 0,9 ≈ −0,105", "a4a0b83e30eaad2d95a9f94c2a1b671db2a8c0f708110949b747b05d25411054": "e^{−5,3} vaut un peu plus de 0,005"}}
📝 Ex 22.Q6b : réponse enregistrée (str

**22.Q7 — Une porte, c'est une multiplication**

In [9]:
wb.record("22.Q7a", 20, mistakes={"multiplie par 0,25, ne divise pas": 320, "une porte multiplie la valeur, elle ne la remplace pas": 80})
wb.record("22.Q7b", "B", mistakes={"la tanh va de −1 à 1": "A", "la ReLU n'est pas bornée": "C",
          "la softmax normalise un vecteur entier": "D"})
wb.record("22.Q7c", False, mistakes={"une porte est entre 0 et 1 : que fait-elle au signe ?": True})
wb.record("22.Q7d", False, mistakes={"un facteur inférieur à 1 peut-il agrandir ?": True})
wb.record("22.Q7e", "A", mistakes={"le learning rate n'intervient qu'à l'entraînement": "B",
          "la loss n'est connue qu'après la sortie": "C", "relis l'équation d'une porte : rien n'y est tiré au hasard": "D"})

WB_ANSWER {"hash": "9363076f914b278eb0f4ee7fb05cf32949e49c1cd61a5c014db357c532254662", "id": "22.Q7a", "kind": "int", "magnitude": 1, "mistakes": {"e21bcdce96649f175d0f4e5764817ecaa069c27a8b9afe6b1d22d6b16e7bbbca": "multiplie par 0,25, ne divise pas", "ffb718033a8c9062d1a0f09f4609ae54bd8e670728a56504405a61c43465274d": "une porte multiplie la valeur, elle ne la remplace pas"}, "sign": 1}
📝 Ex 22.Q7a : réponse enregistrée (int).
WB_ANSWER {"hash": "75d60c74f447b602c5d20ef9458abca9cb85c65dcb5d3077e38d301ff7a692b7", "id": "22.Q7b", "kind": "str", "mistakes": {"46c7409c8ef5a4a3ab75e25006c70452312cc7cd15484f3e9d94ee3af2dd71ce": "la tanh va de −1 à 1", "4f86d16a4720374431033b07580dfd36d295bd99bf15952f1bbac506b58e1255": "la softmax normalise un vecteur entier", "6b03d697e60ba5c7e862072048651614c5bb3016f7c6ecc100d3c8f85d606bbf": "la ReLU n'est pas bornée"}}
📝 Ex 22.Q7b : réponse enregistrée (str).
WB_ANSWER {"hash": "1fc97f062086f79114bd00bf8c49c085e065ba05d55edd1590944ffe103bcbfd", "id": "22.Q

**22.Q8 — Oublier, se souvenir, sélectionner**

In [10]:
wb.record("22.Q8a", "B", mistakes={"« se souvenir », c'est faire entrer du neuf dans la mémoire": "A",
          "la sortie choisit ce qu'on montre, pas ce qu'on garde": "C", "le candidat est la valeur proposée, pas la porte": "D"})
wb.record("22.Q8b", "C", mistakes={"« sélectionner » choisit ce que voit le reste du réseau": "A",
          "la porte d'entrée dose ce qu'on ajoute à la mémoire": "B", "le candidat n'est pas une porte": "D"})
wb.record("22.Q8c", False, mistakes={"relis l'histoire des portes au §22.7.2 : 1997 ou 2000 ?": True})
wb.record("22.Q8d", "B", mistakes={"avec f ≈ 0, la mémoire est effacée": "A", "avec i ≈ 1, on ajoute du neuf à chaque pas": "C",
          "la porte de sortie ne touche pas la mémoire": "D"})
wb.record("22.Q8e", "C", mistakes={"le bloc 0 est i : compte à partir de 0": "A", "i, f, g, o : le bloc 1 est f": "B",
          "i, f, g, o : le bloc 3 est o": "D"})

WB_ANSWER {"hash": "420757484b3d81aa332a267c6b8e3cecd1a8439f69d2dfa7f74194e2a85f9cd8", "id": "22.Q8a", "kind": "str", "mistakes": {"1e2b426b4f65f0539fd6cf1e6563edaebf04d321875eb6620f403fa844219e9b": "la sortie choisit ce qu'on montre, pas ce qu'on garde", "427aa000da2ac9716b3c4d6ff7687f3570cb33f2c8e3cf42a74f096685de55c1": "« se souvenir », c'est faire entrer du neuf dans la mémoire", "894d448d60011c009e3e24844718f42fa411490a03289fab861c414752347785": "le candidat est la valeur proposée, pas la porte"}}
📝 Ex 22.Q8a : réponse enregistrée (str).
WB_ANSWER {"hash": "792eb2c74a38e23a7048b606b0755123175d53b505531df2e51623f37b7b3399", "id": "22.Q8b", "kind": "str", "mistakes": {"27e90b6bd831530f77f4866ad21dd858ffbd666ccf04a53c408ae4f03e99ecd3": "le candidat n'est pas une porte", "4c8c113bead343a30adc9eb8dbe659e2c1d4fd3287393b22e3aa93fc0681cfb9": "la porte d'entrée dose ce qu'on ajoute à la mémoire", "6d87af08675402937c722723fe2b017f773ef8101ac1d3d92ed1243e7ad87de5": "« sélectionner » choisit 

**22.Q9 — LSTM ou GRU ?**

In [11]:
wb.record("22.Q9a", 3, mistakes={"4 blocs, c'est le LSTM": 4, "deux portes, plus le candidat n": 2})
wb.record("22.Q9b", False, mistakes={"cherche une variable c_t dans les équations du GRU": True})
wb.record("22.Q9c", "B", mistakes={"4 blocs pour le LSTM, 3 pour le GRU": "A", "compare les facteurs 3 et 4": "C"})
wb.record("22.Q9d", False, mistakes={"relis le paragraphe « LSTM ou GRU ? »": True})

WB_ANSWER {"hash": "34395d8a81c24a85e9b3d48298ff732f5163fc5be8c13515d3d142b9faf29682", "id": "22.Q9a", "kind": "int", "magnitude": 0, "mistakes": {"40a6b07d8902b195cdd58e6a3b082253faee9bd2da5a0e0a2f0f7f58ed033c71": "4 blocs, c'est le LSTM", "56bb5e82e82363ffa0f04463f64abb722cc9726cc7749d8bee8c2a022c21cca5": "deux portes, plus le candidat n"}, "sign": 1}
📝 Ex 22.Q9a : réponse enregistrée (int).
WB_ANSWER {"hash": "f15f2704c773b5d4dc76abbbc368773ecccd0377a80436b241a450c668a43e15", "id": "22.Q9b", "kind": "bool", "mistakes": {"b56cf15997ad4577b49ac13d94b58ecf628023304883c540e7ff5bb37bec02fb": "cherche une variable c_t dans les équations du GRU"}}
📝 Ex 22.Q9b : réponse enregistrée (bool).
WB_ANSWER {"hash": "4288237fe754a836cd6111906b251a67766eef7da3557d67d661cf87c0f8fb58", "id": "22.Q9c", "kind": "str", "mistakes": {"2cde2a2b79902e7fef71b0849f9791b81333d4a6eac185204cc9b9e17d38f4a9": "4 blocs pour le LSTM, 3 pour le GRU", "a4285edada382b2aa4876ef115989f8580384207143d3db4f7bdb37584eada80": 

**22.Q10 — One-to-many, many-to-one, many-to-many**

In [12]:
wb.record("22.Q10a", "B", mistakes={"une critique entière, un seul verdict": "A", "un verdict pour toute la critique, pas par mot": "C",
          "il n'y a qu'une sortie": "D"})
wb.record("22.Q10b", "A", mistakes={"une seule entrée, une séquence en sortie": "B", "une seule note en entrée": "C",
          "une seule note en entrée": "D"})
wb.record("22.Q10c", "D", mistakes={"une phrase entière en entrée": "A", "une phrase entière en sortie": "B",
          "peut-on traduire le premier mot sans avoir lu la fin de la phrase ?": "C"})
wb.record("22.Q10d", "C", mistakes={"une vidéo en entrée": "A", "un label par image, pas un pour toute la vidéo": "B",
          "chaque image a son label au même instant : pas besoin d'attendre la fin": "D"})
wb.record("22.Q10e", "B", mistakes={"40 caractères en entrée": "A", "une seule sortie par fenêtre": "C", "une seule sortie par fenêtre": "D"})

WB_ANSWER {"hash": "b46daacbda9ef836d70a8676ed3594fa0e032bd905e2ef0b64ee2a99fb147c2c", "id": "22.Q10a", "kind": "str", "mistakes": {"188f75221c62491a45e67b1a35d15bc8b15fda9a35df6386ef68f4b9015a21c7": "un verdict pour toute la critique, pas par mot", "952c590bbd8c0bdfeb4ded45bd545e8e18b8d27d2aeb2c76702adb95d765de63": "il n'y a qu'une sortie", "c9be5fd148b96e51c855a816a3a766edd4850e7beb6d56b500d3314627d34a0b": "une critique entière, un seul verdict"}}
📝 Ex 22.Q10a : réponse enregistrée (str).
WB_ANSWER {"hash": "73cc0407880b14ac92e2a15112a82cec22a8b75387fe7bcc012ecbda92ecdef3", "id": "22.Q10b", "kind": "str", "mistakes": {"62bddc8d64684ce8dfc2aa06390ba8faf140efeacadd8b5f15ac0bd5f802f35a": "une seule entrée, une séquence en sortie", "e26cfdcfdafd21f8e1262820f16b57af90a9f781145f6b53ad31ad989bf0ee2e": "une seule note en entrée"}}
📝 Ex 22.Q10b : réponse enregistrée (str).
WB_ANSWER {"hash": "9a0c8cad14959ed25458a4865066d4adeb1e96b572b34c818bf2ef21336be107", "id": "22.Q10c", "kind": "str", "m

**22.Q11 — RNN profond et bidirectionnel : quand et à quel prix ?**

In [13]:
wb.record("22.Q11a", False, mistakes={"le sens arrière a besoin de la fin de la séquence : l'a-t-on en temps réel ?": True})
wb.record("22.Q11b", "B", mistakes={"une couche RNN lit une séquence, pas une valeur": "A", "les couches ne s'échangent pas leurs poids": "C",
          "relis le §22.8.2": "D"})
wb.record("22.Q11c", "C", mistakes={"une somme garderait H valeurs par pas, or output en a 2H": "A",
          "une moyenne garderait H valeurs par pas, or output en a 2H": "B", "un produit garderait H valeurs par pas": "D"})
wb.record("22.Q11d", True, mistakes={"compte les paramètres d'une couche bidirectionnelle (weight_ih_l0_reverse…)": False})
wb.record("22.Q11e", "B", mistakes={"chaque couche et chaque sens ont leurs propres poids": "A", "il s'entraîne toujours par BPTT": "C",
          "rien ne limite la longueur des séquences": "D"})

WB_ANSWER {"hash": "da22c6b3684a2b0e7407c60e2d89d509a2c6a87751cd177018c3bd6ef45acb80", "id": "22.Q11a", "kind": "bool", "mistakes": {"74bff389449d228c3d9f081f545af80074f9ec651deb6671a03f10e3a027b53e": "le sens arrière a besoin de la fin de la séquence : l'a-t-on en temps réel ?"}}
📝 Ex 22.Q11a : réponse enregistrée (bool).
WB_ANSWER {"hash": "8fb6eb998f5b051e9dc25ef5018f6b63e491421be92ce97ef11b432f9211215a", "id": "22.Q11b", "kind": "str", "mistakes": {"2b57f5fcbe6cd706d382f0503c4e65076c20140bb752953750ac3b3b6cc3c1f4": "relis le §22.8.2", "501df621ac819a21f3be7d141d0cd6d0b42f5dc62752e0ed50b2aea26fa3f9db": "une couche RNN lit une séquence, pas une valeur", "7b9cab3a2bd745082dfe73bd0edee82b86de7b533b77deea2dc0b4512ba06884": "les couches ne s'échangent pas leurs poids"}}
📝 Ex 22.Q11b : réponse enregistrée (str).
WB_ANSWER {"hash": "63395d5b28a77b6aed2e1292bbeb8a6b504a599fdd7d6c675153a8107f70285b", "id": "22.Q11c", "kind": "str", "mistakes": {"073f7f2fc64261ff292a7c77408e08188b03a546b7ff27

**22.Q12 — 🕰️ Transformers, RNN légers et modèles à espace d'états**

In [14]:
wb.record("22.Q12a", 2017, mistakes={"1997, c'est le LSTM": 1997, "2014, c'est le GRU": 2014})
wb.record("22.Q12b", True, mistakes={"un RNN peut-il calculer h_t avant h_{t−1} ?": False})
wb.record("22.Q12c", "B", mistakes={"les grands modèles de langage sont des Transformers": "A",
          "pour de très longs textes, l'attention l'emporte": "C", "la génération d'images n'est pas une affaire de RNN": "D"})
wb.record("22.Q12d", "B", mistakes={"relis la section « Après les RNN »": "A", "un SSM est un réseau de neurones": "C",
          "un modèle à espace d'états a, par définition, un état": "D"})
wb.record("22.Q12e", True, mistakes={"relis la première règle de la section sur les séries temporelles": False})

WB_ANSWER {"hash": "b732e37a3349c57234552e6c82ed63be9f94062e0643504fa136a256fc08b81d", "id": "22.Q12a", "kind": "int", "magnitude": 3, "mistakes": {"9bf6aab617e2a5b87d3a7af901e41228b8182c4a3369240e91a6fa75f2334dc6": "1997, c'est le LSTM", "fc6e4fd7a9d4d574490f3e7f60156481989dcbb557115d2fc0d07f1498ce441f": "2014, c'est le GRU"}, "sign": 1}
📝 Ex 22.Q12a : réponse enregistrée (int).
WB_ANSWER {"hash": "f92a431240dad3f92fe482328b7d4454886a469888c84401ebf7fe506def62d0", "id": "22.Q12b", "kind": "bool", "mistakes": {"dff2adb37db95d35ba2acbe893751361121f8e0701cf4f01cfd68b749203a818": "un RNN peut-il calculer h_t avant h_{t−1} ?"}}
📝 Ex 22.Q12b : réponse enregistrée (bool).
WB_ANSWER {"hash": "6e018df63368fe67be87b2bcb63d8621a09217f6d7774824ea01f8cd99414c0c", "id": "22.Q12c", "kind": "str", "mistakes": {"2aa2132fdcb0cb2ce63c43d1909ff001d4f9b46c2aed524d1342e01a5d456ca4": "les grands modèles de langage sont des Transformers", "7cbee73d96f488592a826e3e682efe1e05c1691dda750611c112386dd7951f7f": "l

**22.R1 — Une convolution 1D peut-elle lire une séquence ?**

In [15]:
wb.record("22.R1a", 20 - 4 + 1, mistakes={"n − k + 1 : n'oublie pas le + 1": 16, "sans padding, la sortie raccourcit": 20})
wb.record("22.R1b", True, mistakes={"relis le partage de poids (ch. 21)": False})
wb.record("22.R1c", 3 + 2, mistakes={"chaque couche ajoute k − 1 = 2": 6, "3 + 2, pas 3 × 3": 9})
wb.record("22.R1d", 500, mistakes={"le champ de n couches vaut 2n + 1 : résous 2n + 1 ≥ 1 001": 1000,
          "2n + 1 ≥ 1 001 : n = 500 suffit": 501, "chaque couche ajoute 2, pas 3": 334})

WB_ANSWER {"hash": "2c1694bd18964211b5f4ce3efc36460a9aa5efbd3de6ff0c40562e3296d3d3e4", "id": "22.R1a", "kind": "int", "magnitude": 1, "mistakes": {"a9bf0853bdfc4e273229d61043784154b4dc2a8a6b6a939d89ea7e876c31b761": "sans padding, la sortie raccourcit", "c62a05b21a47f68d16d28c7e87355e16b72f896f7980a3eba1736a1544f32377": "n − k + 1 : n'oublie pas le + 1"}, "sign": 1}
📝 Ex 22.R1a : réponse enregistrée (int).
WB_ANSWER {"hash": "ddbfe9f97b54f05347d0c4516bc5e1b27f1f09518feeb42937cae7b53e5d5a94", "id": "22.R1b", "kind": "bool", "mistakes": {"582b9234b5cdbd5d6bf7fc01794bd224dbfcfe43c6b68a6d1c7fcd6043826cee": "relis le partage de poids (ch. 21)"}}
📝 Ex 22.R1b : réponse enregistrée (bool).
WB_ANSWER {"hash": "fa44519d315c77e1c1be08d0db57d35d33d380af5a9921fd6730d3a1ee1f83d6", "id": "22.R1c", "kind": "int", "magnitude": 0, "mistakes": {"0712877ecf929f67edbc0b8679109dad33e91ef6157f12365858afe25baf1d53": "3 + 2, pas 3 × 3", "d352b5040ccbde8e5fb2c9bd8b14517e61407f5a273d91bb4ead6eef6e9f2bb9": "chaque

**22.R2 — Gradient clipping et optimiseurs adaptatifs**

In [16]:
wb.record("22.R2a", 5, mistakes={"la norme euclidienne, pas la somme": 7})
wb.record("22.R2b", 3 / math.hypot(3, 4), decimals=2, mistakes={"la norme dépasse 1 : il faut réduire le gradient": 3, "divise par la norme, pas par 4": 0.75})
wb.record("22.R2c", False, mistakes={"on multiplie tout le vecteur par un même nombre positif": True})
wb.record("22.R2d", "A", mistakes={"RMSprop regarde les gradients, pas les poids": "B", "le pas ne dépend pas de l'epoch": "C",
          "relis la définition de RMSprop (ch. 19)": "D"})
wb.record("22.R2e", True, mistakes={"écris le premier pas d'Adam : g apparaît en haut et en bas": False})

WB_ANSWER {"hash": "97fff806db99ddb9d23065059f0eddd48a99e8abb28ead08b62a59c1642e55d0", "id": "22.R2a", "kind": "int", "magnitude": 0, "mistakes": {"ad9e7906fdb33516b7f84c5366bfecf31f3e381a099bc34e2c9e2a5436b10d41": "la norme euclidienne, pas la somme"}, "sign": 1}
📝 Ex 22.R2a : réponse enregistrée (int).
WB_ANSWER {"decimals": 2, "hash": "15d485039702b68a0a43274517a0541a2e75de65736e76e3abd139e98a57ca18", "hash_coarse": "50447e75748536b3a2da0cf09945e8cc0db8cd14a9d8b4381dffa4e46d020209", "id": "22.R2b", "kind": "float", "magnitude": -1, "mistakes": {"0edb51059fb9fbe8c259015564b1728cb6bc23752a69073749271f0999e12582": "la norme dépasse 1 : il faut réduire le gradient", "28bd1b212b7739451cb9938f4141f5a31924354334dbf95c3d4c12baa8ea86ae": "divise par la norme, pas par 4"}, "sign": 1}
📝 Ex 22.R2b : réponse enregistrée (float, 2 décimale(s)).
WB_ANSWER {"hash": "7e20d9dd75f98bb70e8ca865ff332f3015befaca892df0fd5baf2bea0726bf2a", "id": "22.R2c", "kind": "bool", "mistakes": {"707ff5b7f2967e132bad3

**22.R3 — Ajuster le scaler sur le train seulement**

In [17]:
wb.record("22.R3a", False, mistakes={"le test doit venir après l'entraînement, comme en production": True})
wb.record("22.R3b", "B", mistakes={"ajuster sur les 100 valeurs fait entrer le test dans le scaler": "A",
          "le scaler s'ajuste sur l'entraînement": "C", "un seul scaler pour toute la série": "D"})
wb.record("22.R3c", (250 - 0) / (200 - 0), decimals=2, mistakes={"le scaler ne coupe pas les valeurs à 1": 1.0, "(x − min)/(max − min), pas l'inverse": 0.8})
wb.record("22.R3d", False, mistakes={"en production, réajusterais-tu le scaler sur chaque nouvelle valeur ?": True})
wb.record("22.R3e", "transform", mistakes={"on ne réajuste pas le scaler sur le test": "fit_transform", "fit ajuste ; il faut appliquer": "fit"})

WB_ANSWER {"hash": "8ed88bf497581eb19359507fbd7538593f0266976d4c81e4c47b221a75b31630", "id": "22.R3a", "kind": "bool", "mistakes": {"df63134c271ce0a6e37ee8d2ded9dbe00e50a89b62739ce357aaada38cdc2b8e": "le test doit venir après l'entraînement, comme en production"}}
📝 Ex 22.R3a : réponse enregistrée (bool).
WB_ANSWER {"hash": "f7e5772c387d6cec2192e803501dfdae9353e86e54da8b40e24e1cd83825fd7e", "id": "22.R3b", "kind": "str", "mistakes": {"2213f5d6f32073714b1e8e0d357a469c93838ce2902348849a4d6643d8e1131b": "ajuster sur les 100 valeurs fait entrer le test dans le scaler", "a843975f5fa92d3817e3a6ddfed07c9bd6aceec6dc3b39715fa96129cc89d6ed": "un seul scaler pour toute la série", "dfeb1f96c1bcd7bab523b03a95f914837869b13e6ac6582ddacea890918612b1": "le scaler s'ajuste sur l'entraînement"}}
📝 Ex 22.R3b : réponse enregistrée (str).
WB_ANSWER {"alt_coarse": ["16a0660f65e6e475a130ba69aa9474ff591909d84cbff8b97f33b53539238ef4"], "decimals": 2, "hash": "70c3cccab126fff62e957dba50c40406f4f71d4894fac0c54d3d

**Ex 22.1 — Combien de fenêtres ? Températures, taches solaires et Holmes**

In [18]:
wb.record("22.1a", n_windows(24, 4, 1, 1), mistakes={"la cible doit exister : n'oublie pas l'horizon": 21, "n'oublie pas le + 1": 19})
wb.record("22.1b", n_windows(24, 4, 1, 2), mistakes={"⌊19/2⌋ + 1": 9})
wb.record("22.1c", (2024 - 1749 + 1) * 12, mistakes={"de 1749 à 2024 inclus, il y a 276 années": 275 * 12, "en mois, pas en années": 276})
wb.record("22.1d", n_windows(3312, 120, 6, 1), mistakes={"l'horizon h = 6 compte": 3193, "n'oublie pas le + 1": 3186})
wb.record("22.1e", n_windows(3312 - 600, 120, 6, 1), mistakes={"la cible aussi doit être avant le test : l'horizon compte": 2593, "en fenêtres, pas en mois": 2712})
wb.record("22.1f", n_windows(575_000, 40, 1, 3), mistakes={"une fenêtre tous les 3 caractères": 574_960, "n'oublie pas le + 1": 191_653})
wb.record("22.1g", "C", mistakes={"N ≈ T/s : inverse la formule, T ≈ N × s": "A", "chaque fenêtre démarre 3 caractères après la précédente": "B",
          "un demi-million × 3": "D"})

WB_ANSWER {"hash": "9587d8a1fbcad32a6e97b263c187cde95414240bf35e4a5e643cfe2aaa43f564", "id": "22.1a", "kind": "int", "magnitude": 1, "mistakes": {"40eb74e42aaf85662a6d567657f25feb2e17cf64ce5d5930250de76103118bec": "la cible doit exister : n'oublie pas l'horizon", "e8c914f54e5e4ec2b657ec83968c62eb4a347f23a0de590f591df96541d35608": "n'oublie pas le + 1"}, "sign": 1}
📝 Ex 22.1a : réponse enregistrée (int).
WB_ANSWER {"hash": "4da5dc54b07c2a354c4705da747cb3cd8ce32b58527747dfd280a35093aa869e", "id": "22.1b", "kind": "int", "magnitude": 1, "mistakes": {"6d78379def8df3ab29e786ad56bc8bcc67b3cf2ad0987c163a47292e777239c5": "⌊19/2⌋ + 1"}, "sign": 1}
📝 Ex 22.1b : réponse enregistrée (int).
WB_ANSWER {"hash": "35b6937f1c07ec06002dc1dd297f19cbd8a49408615f7f1ff0578bc92d05c9b1", "id": "22.1c", "kind": "int", "magnitude": 3, "mistakes": {"67195a40e59f9af4374955a19ee0a9f198ebe2b4de2ca472ae012067c4f22787": "en mois, pas en années", "89065f6298c6965012bef8b316557ecdbee66b3aeb1357c45b72e31f8b72a51c": "de 1

**Ex 22.2 — Dérouler à la main une cellule RNN scalaire sur trois pas**

In [19]:
wb.record("22.2a", H_222[0], decimals=4, mistakes={"applique la tanh": 2.0})
wb.record("22.2b", H_222[1], decimals=4, mistakes={"x₂ = 0, mais l'état précédent compte": 0.0})
wb.record("22.2c", H_222[2], decimals=4, mistakes={"ajoute w h₂ à u x₃": math.tanh(-1)})
wb.record("22.2d", 2 * H_222[2], decimals=4, mistakes={"y₃ = v h₃ avec v = 2": H_222[2]})
wb.record("22.2e", H_REV_222[2], decimals=4, mistakes={"n'oublie pas l'état h₂ dans le dernier pas": math.tanh(2)})
wb.record("22.2f", 0.5 ** 2 * 1, decimals=2, mistakes={"x₁ traverse deux pas après le sien : w²": 0.5, "x₁ entre au pas 1, et il reste deux pas : w², pas w³": 0.125})

WB_ANSWER {"decimals": 4, "hash": "b7a08b2a842c613dc84363d99a47ee351ff2255bd56eb2680a5b34d818698564", "hash_coarse": "72eca5a2dae0172f3b52f87a7a54cf2e153c2330d777964da55ecc3f73bf96c5", "id": "22.2a", "kind": "float", "magnitude": -1, "mistakes": {"08eb28c19dec52e7c9c9314917b818e83888f3332b53cf693b8d4356f34fee0d": "applique la tanh"}, "sign": 1}
📝 Ex 22.2a : réponse enregistrée (float, 4 décimale(s)).
WB_ANSWER {"decimals": 4, "hash": "cc0c01badac905c6ace356e2d8728ab35aedd44d9d7a76951f89f37e78551c1d", "hash_coarse": "783fc37682311cb1765903e7b5410abff93a14c9ad8d039109ae89b18f9ce396", "id": "22.2b", "kind": "float", "magnitude": -1, "mistakes": {"b69265a8840c2e8298dc40569f0361baba3b1de7dd4b6a439de461478ba50103": "x₂ = 0, mais l'état précédent compte"}, "sign": 1}
📝 Ex 22.2b : réponse enregistrée (float, 4 décimale(s)).
WB_ANSWER {"decimals": 4, "hash": "2dc864118069170485280d0ee5956895038d0297d74542f212e3579cdc32cacf", "hash_coarse": "d14e8f32d9256fa396a1e78a70c4bedb6a0a6289430c426e0c0512

**Ex 22.3 — Compter les poids : la cellule à 69 poids du livre, puis nn.RNN et nn.LSTM**

In [20]:
wb.record("22.3a", 8 * 10, mistakes={"les neurones A n'ont pas de biais": 88, "8 neurones qui lisent 10 entrées : un produit": 18})
wb.record("22.3b", 8 * 8, mistakes={"les neurones B n'ont pas de biais": 72, "8 neurones qui lisent 8 valeurs : un produit": 16})
wb.record("22.3c", 3 * 8 + 3, mistakes={"les neurones C ont un biais": 24})
wb.record("22.3d", 80 + 64 + 8 + 27, mistakes={"n'oublie pas les 8 biais de l'état": 171, "ajoute les neurones de sortie C": 152})
wb.record("22.3e", n_recurrent(10, 8, 1), mistakes={"PyTorch a deux vecteurs de biais, b_ih et b_hh": 152, "n'oublie pas les biais": 144})
wb.record("22.3f", n_recurrent(10, 8, 1) + 8 * 3 + 3 - 179, mistakes={"l'écart, pas le total": 187, "PyTorch compte un vecteur de biais de plus": 0})
wb.record("22.3g", n_recurrent(10, 8, 4), mistakes={"4 blocs de portes": 160, "PyTorch a deux vecteurs de biais par couche": 608})
wb.record("22.3h", n_recurrent(89, 128, 4) + n_recurrent(128, 128, 4) + 128 * 89 + 89, mistakes={"la deuxième couche LSTM reçoit 128 valeurs, pas 89": 2 * n_recurrent(89, 128, 4) + 128 * 89 + 89,
          "compte comme PyTorch : deux vecteurs de biais par couche LSTM": 254_681,
          "n'oublie pas la couche dense": n_recurrent(89, 128, 4) + n_recurrent(128, 128, 4)})
wb.record("22.3i", 2 * 4 * 128, mistakes={"il y a deux couches LSTM": 512, "un vecteur de biais a 4 × 128 valeurs (une par porte)": 256})

WB_ANSWER {"hash": "6fe044b0f699fb0e9c18d2a223d9dafb8b2266adffe6457e30d49f3e56b8200e", "id": "22.3a", "kind": "int", "magnitude": 1, "mistakes": {"90d0de4242aedbc53feaaf04a9a50144da54509e83f4f47cb9ac483197d29230": "8 neurones qui lisent 10 entrées : un produit", "d2c6e2c1a315541624170a1e3af7ac56d6e12eaa14cfa40a7b39c0ce23511904": "les neurones A n'ont pas de biais"}, "sign": 1}
📝 Ex 22.3a : réponse enregistrée (int).
WB_ANSWER {"hash": "30626a8d077f0cc9ce3686a269da968c05c8ddbf91675e6f0e16035324e51b1a", "id": "22.3b", "kind": "int", "magnitude": 1, "mistakes": {"607627c3df2dedfddd5c49e3a52bdcf57d59187301cd629ae79149b78db5b640": "8 neurones qui lisent 8 valeurs : un produit", "d8c44d00229251dd833f127f5e1b8aae2d69f38b9609feb84514cf699f7866d7": "les neurones B n'ont pas de biais"}, "sign": 1}
📝 Ex 22.3b : réponse enregistrée (int).
WB_ANSWER {"hash": "a8aed1a7907a147e095baac921c037875b8696254b1d163241c2b007aa2d5bd3", "id": "22.3c", "kind": "int", "magnitude": 1, "mistakes": {"5e5c1c8f7745dd

**Ex 22.4 — Un pas de LSTM à la main**

In [21]:
wb.record("22.4a", I_224, decimals=4, mistakes={"σ(0) = 0,5, pas 0": 0.0, "n'oublie pas W_hh h_{t−1}": sigmoid(-0.5)})
wb.record("22.4b", F_224, decimals=4, mistakes={"le biais de f vaut 1": 0.5, "attention au signe de W_hh (−2)": sigmoid(3)})
wb.record("22.4c", G_224, decimals=4, mistakes={"le candidat g passe par une tanh, pas une sigmoïde": sigmoid(1)})
wb.record("22.4d", O_224, decimals=4, mistakes={"n'oublie pas W_hh h_{t−1} = 3 × 0,5": sigmoid(-1)})
wb.record("22.4e", C_224, decimals=4, mistakes={"la porte d'oubli multiplie c_{t−1}": 2 + I_224 * G_224, "ajoute i g": F_224 * 2})
wb.record("22.4f", H_224, decimals=4, mistakes={"h_t = o tanh(c_t) : n'oublie pas la tanh": O_224 * C_224, "multiplie par la porte de sortie": math.tanh(C_224)})
wb.record("22.4g", C_OPEN_224, decimals=4, mistakes={"σ(−20) ≈ 0 : la porte d'entrée est fermée": 2 + G_224})
wb.record("22.4h", F_224, decimals=4, mistakes={"la dérivée de f c_{t−1} par rapport à c_{t−1} n'est pas 1": 1.0})
wb.record("22.4i", "(4, 1)", mistakes={"PyTorch range les 4 blocs dans les LIGNES de weight_ih_l0 : (4H, D)": "(1, 4)",
          "4 blocs de portes": "(1, 1)"})

WB_ANSWER {"decimals": 4, "hash": "e806cd9bdba8a37943e4e4ac57ef6d73cf7a41eb2b0da754c2186a313eade49d", "hash_coarse": "df90642484a7fcf9c04ee468b5748d91b536c38b043bf26b7314e634dc0a92d4", "id": "22.4a", "kind": "float", "magnitude": -1, "mistakes": {"a9dae256f734309c3dfe3b26ff63876db752b5f7edca378defb63fd6a2f33ed9": "n'oublie pas W_hh h_{t−1}", "c2ed18a81d14acc0726ceca5e001e4483a1e8cf43a9053b732a73ec522dd4034": "σ(0) = 0,5, pas 0"}, "sign": 1}
📝 Ex 22.4a : réponse enregistrée (float, 4 décimale(s)).
WB_ANSWER {"decimals": 4, "hash": "e460fb87991489fbea6141705752ee5b5aa29409ea1f460c7147d72c0ac03901", "hash_coarse": "d568196f25c28141df5387c2c05d8205c6d391abec42f9e7b6b08e46f3044419", "id": "22.4b", "kind": "float", "magnitude": -1, "mistakes": {"261c867d887b4bceffbdb1ead790189d0e8c9a77ccd57d8f1dff0332ccd0b20c": "le biais de f vaut 1", "ef270108a78ef040d55b316396a941294389a754fd7a99428d4209946028b857": "attention au signe de W_hh (−2)"}, "sign": 1}
📝 Ex 22.4b : réponse enregistrée (float, 4 d

**Ex 22.5 — Softmax avec température à la main**

In [22]:
wb.record("22.5a", softmax_t(Z_225, 1)[0], decimals=4, mistakes={"la softmax prend des exponentielles, pas les logits eux-mêmes": 3 / 4.5})
wb.record("22.5b", softmax_t(Z_225, 0.5)[0], decimals=4, mistakes={"n'oublie pas le troisième caractère": 1 / (1 + math.exp(-4))})
wb.record("22.5c", softmax_t(Z_225, 2)[0], decimals=4, mistakes={"n'oublie pas le troisième caractère": 1 / (1 + math.exp(-1))})
wb.record("22.5d", 1.0, decimals=4, mistakes={"quand T tend vers 0, le plus probable prend toute la probabilité": 0.0})
wb.record("22.5e", 1 / 3, decimals=4, mistakes={"quand T tend vers l'infini, tous les logits divisés par T tendent vers 0": 0.0})
wb.record("22.5f", math.exp((3 - 1) / 0.5), decimals=2, mistakes={"à T = 0,5, l'écart 3 − 1 est divisé par 0,5": math.exp(2), "le rapport des probabilités, pas des logits": 3.0})
wb.record("22.5g", True, mistakes={"diviser tous les logits par un même T > 0 change-t-il leur ordre ?": False})

WB_ANSWER {"decimals": 4, "hash": "e0a9493784b7b7be18452fbb669807f288466d6d53574b8325f1f03326a6be34", "hash_coarse": "8e5cdfd71b900be66befc6c6d8d9bb597a16ca254e45beffb26424cf1680e496", "id": "22.5a", "kind": "float", "magnitude": -1, "mistakes": {"8319b801e64080c73de0f75122063cb3240d3eccde5bec499a31a7061807f9ac": "la softmax prend des exponentielles, pas les logits eux-mêmes"}, "sign": 1}
📝 Ex 22.5a : réponse enregistrée (float, 4 décimale(s)).
WB_ANSWER {"decimals": 4, "hash": "98d1b59ea6d73751fa106e751fdfdabcb4cebd9d6b07d670d4bbd98a990ced3b", "hash_coarse": "5f6b670672c7ef9ecfb12db4773153fa1436f164f41806dbaabd41fe5fff40af", "id": "22.5b", "kind": "float", "magnitude": -1, "mistakes": {"6f9f5a6c2761f2dbd80a7add0c566dfe9693e9f6472c60551afe4252f26b207e": "n'oublie pas le troisième caractère"}, "sign": 1}
📝 Ex 22.5b : réponse enregistrée (float, 4 décimale(s)).
WB_ANSWER {"decimals": 4, "hash": "7b77536e08a5954a2e4e95626094e57a400aff608c9a7e30a22be5680d4bbc26", "hash_coarse": "b1b75367b0

**Ex 22.8 — Fermi : les 17 millions de poids du CNN-LSTM et le prix d'une epoch**

In [23]:
wb.record("22.8a", D_228, mistakes={"le stride 2 divise chaque côté par 2": 128 * 128 * 8, "n'oublie pas les 8 canaux": SIDE_228 ** 2})
wb.record("22.8b", CONV_228, mistakes={"un biais par filtre": 8 * 25, "les images n'ont qu'un canal": 8 * (25 * 3 + 1)})
wb.record("22.8c", LSTM_228, mistakes={"4 blocs de portes": n_recurrent(D_228, 128, 1),
          "deux vecteurs de biais de 4 × 128 valeurs": 4 * (D_228 * 128 + 128 * 128)})
wb.record("22.8d", DENSE_228, mistakes={"n'oublie pas les 9 biais": 128 * 9})
wb.record("22.8e", CONV_228 + LSTM_228 + DENSE_228, mistakes={"additionne les trois couches": LSTM_228})
wb.record("22.8f", n_recurrent(D_228, 32, 4), mistakes={"4 blocs de portes": n_recurrent(D_228, 32, 1)})
wb.record("22.8g", MACS_228, mistakes={"4 blocs de 128 colonnes : D × 4 × 128": D_228 * 128})
wb.record("22.8h", int(math.log10(MACS_228 * 1000 * 16)), mistakes={"pour toute l'epoch : 1 000 vidéos de 16 images": 7, "1 000 × 16 = 16 000 images, pas 1 000": 10})

WB_ANSWER {"hash": "260a7d79b2245aa2e156c0da0d52c0f07db2fa2508b47d014a6f37b8a66ce365", "id": "22.8a", "kind": "int", "magnitude": 4, "mistakes": {"caa31e3978f300524c39a6edb97ab630e70ad26d3474dd954123806067717828": "n'oublie pas les 8 canaux", "e6074e1d23a2b270e98fa0ceca5d880b5150ac25cad2b9a3206b3619db7836e1": "le stride 2 divise chaque côté par 2"}, "sign": 1}
📝 Ex 22.8a : réponse enregistrée (int).
WB_ANSWER {"hash": "165c76c266e7c5339a826a8153315967ff0c05a9820ec93a681835659bd3f5ad", "id": "22.8b", "kind": "int", "magnitude": 2, "mistakes": {"9e10d177fdfa6a7d8c797eb821075971aba7c45082efd0e0e05dd16af2de109d": "un biais par filtre", "b605f3f8376ed801148240e5c8c2043bee386ccb0b51dbd09033a190533f8a9f": "les images n'ont qu'un canal"}, "sign": 1}
📝 Ex 22.8b : réponse enregistrée (int).
WB_ANSWER {"hash": "7ffca0930eb6d85db5ffac165f770b4d5e451356454ec227688e191704e0cee1", "id": "22.8c", "kind": "int", "magnitude": 7, "mistakes": {"7c9c71b33a9d23e5c6d8ba592518d7e18df4cac4f016e0c9c68596c0769cb

## ✅ Bilan

**Auto-évaluation** (note-toi de 0 à 3 dans `mon_travail/suivi/auto_evaluation.md`) :
1. Sais-tu écrire les équations d'une cellule RNN, LSTM et GRU, et compter leurs paramètres ?
2. Sais-tu expliquer pourquoi le gradient d'un RNN s'évanouit ou explose, et ce que change le chemin additif du LSTM ?
3. Sais-tu préparer une série temporelle sans fuite, et régler la température d'un générateur ?

**Pour aller plus loin** : le billet de C. Olah sur les LSTM et celui d'A. Karpathy sur les RNN, cités dans la fiche. La suite : le ch. 23, Keras, traduit en PyTorch.

*Les exercices de code 22.12 à 22.29 (parties A à D, tes librairies `sequences.py` et `rnn.py` et les RNN PyTorch) seront ajoutés à la prochaine session de génération ; relance alors `python tools/start_chapter.py 22` pour obtenir le notebook complet.*